# Pelajaran 10 - Ejen AI dalam Pengeluaran

Dalam pelajaran ini anda akan mempelajari **corak pengeluaran** untuk ejen AI menggunakan Microsoft Agent Framework (Python). Kami merangkumi:

- **Kebolehlihatan** — menambah pengukuran masa dan pencatatan log pada interaksi ejen
- **Penilaian** — menggunakan ejen penilai untuk memberi skor kualiti respons
- **Pengurusan kos** — strategi untuk pengoptimuman token dan pemilihan model

Senario ini adalah sebuah **ejen pelancongan** yang membantu pengguna merancang perjalanan, dengan pemantauan dan penilaian ditambahkan di atasnya.


## Persediaan


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -U -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
import time
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
# Create the Azure AI Foundry provider
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())


## Pertimbangan Pengeluaran

Memindahkan ejen AI daripada prototip ke pengeluaran memerlukan perhatian yang teliti terhadap tiga tonggak:

1. **Observabiliti** — Anda memerlukan keterlihatan terhadap apa yang dilakukan ejen, berapa lama ia mengambil masa, dan alat mana yang dipanggil. Tanpa penjejakan dan pencatatan, penyahpepijatan isu pengeluaran hampir mustahil.

2. **Penilaian** — Pemeriksaan kualiti automatik memastikan respons ejen kekal tepat, lengkap, dan berguna dari masa ke masa. Ejen penilai boleh memberi skor kepada respons berdasarkan kriteria yang ditetapkan.

3. **Pengurusan Kos** — Penggunaan token memberi kesan langsung kepada kos. Strategi seperti pengoptimuman prompt, pemilihan model, dan caching membantu mengekalkan perbelanjaan terkawal tanpa mengorbankan kualiti.


## Membina Ejen yang Boleh Dipantau

Kami mentakrifkan alat perjalanan dan membalut panggilan ejen dengan pengukuran masa supaya kami boleh memantau kelewatan. Dalam persekitaran pengeluaran, anda akan mengintegrasikannya dengan OpenTelemetry atau backend penjejakan yang serupa.


In [ ]:
@tool(approval_mode="never_require")
def get_flight_info(destination: Annotated[str, "The destination city"]) -> str:
    """Get flight information for a destination."""
    flights = {
        "Paris": "BA 304, 08:30-11:45, $350",
        "Tokyo": "JL 044, 11:00-07:00+1, $890",
        "Barcelona": "VY 7821, 07:15-10:30, $280",
    }
    return flights.get(destination, f"No flights found to {destination}")


@tool(approval_mode="never_require")
def get_activity_suggestions(destination: Annotated[str, "The destination city"]) -> str:
    """Get activity suggestions for a destination."""
    activities = {
        "Paris": "Louvre Museum, Eiffel Tower, Seine River Cruise, Montmartre walking tour",
        "Tokyo": "Senso-ji Temple, Tsukiji Market tour, Shibuya Crossing, teamLab Borderless",
        "Barcelona": "Sagrada Familia, Park Güell, La Boqueria Market, Gothic Quarter walk",
    }
    return activities.get(destination, f"No activities found for {destination}")

In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_info, get_activity_suggestions],
    name="TravelAgent",
    instructions="You are a helpful travel agent. Use the available tools to help users plan their trips. Provide comprehensive, actionable travel advice.",
)

# Simple observability: track timing
start_time = time.time()
response = await agent.run(
    "I want to plan a day trip in Paris. What flights and activities do you recommend?",
    )
elapsed = time.time() - start_time
print(f"Response ({elapsed:.2f}s):\n{response}")

## Corak Penilaian

Satu corak pengeluaran yang biasa ialah menggunakan ejen kedua sebagai seorang **penilai**. Penilai memberi skor jawapan ejen utama berdasarkan kriteria yang telah ditetapkan seperti kelengkapan, ketepatan, dan kegunaan.

Ini membolehkan:
- Gerbang kualiti automatik sebelum respons sampai kepada pengguna
- Pengesanan regresi apabila arahan atau model berubah
- Pemantauan berterusan prestasi ejen dari masa ke masa


In [ ]:
evaluator = await provider.create_agent(
    name="ResponseEvaluator",
    instructions="""You evaluate travel agent responses on these criteria:
1. Completeness (1-5): Did it cover flights AND activities?
2. Accuracy (1-5): Is the information consistent?
3. Helpfulness (1-5): Would a traveler find this actionable?
4. Overall Score (1-5)
Provide scores and a brief explanation for each.""",
)

evaluation = await evaluator.run(f"Evaluate this travel agent response:\n\n{response}")
print(f"Evaluation:\n{evaluation}")

## Strategi Pengurusan Kos

Mengawal kos adalah penting untuk ejen AI pengeluaran. Berikut adalah strategi utama:

| Strategy | Description |
|---|---|
| **Pengoptimuman prompt** | Pastikan arahan sistem ringkas. Buang konteks yang berlebihan untuk mengurangkan token input. |
| **Pemilihan model** | Gunakan model yang lebih kecil dan lebih murah (contohnya GPT-4o-mini) untuk tugas mudah seperti pengkelasan atau pengekstrakan, dan simpan model yang lebih besar untuk penalaran yang kompleks. |
| **Caching** | Simpan hasil alat dan pertanyaan kerap untuk mengelakkan panggilan API yang berulang. |
| **Bajet token** | Tetapkan had `max_tokens` untuk mengelakkan respons yang terlalu panjang secara tidak dijangka. |
| **Pengelompokan** | Kumpulkan beberapa pertanyaan pengguna ke dalam satu panggilan API jika boleh. |

Dalam praktiknya, pendekatan berperingkat berkesan: arahkan permintaan yang mudah kepada model yang pantas dan murah, dan eskalasikan hanya pertanyaan yang kompleks kepada model yang lebih berkemampuan.


## Ringkasan

Dalam pelajaran ini, anda telah belajar bagaimana untuk:

1. **Menambah observabiliti** kepada interaksi ejen dengan penjejakan masa dan pencatatan, yang meletakkan asas untuk penjejakan dan pemantauan.
2. **Menilai respons ejen** secara automatik menggunakan ejen penilai yang memberikan skor untuk kelengkapan, ketepatan, dan kegunaan.
3. **Mengurus kos** melalui pengoptimuman prompt, pemilihan model, penyimpanan cache, dan bajet token.

Corak pengeluaran ini membantu memastikan ejen AI anda boleh dipercayai, boleh diukur, dan efektif dari segi kos pada skala.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Penafian:
Dokumen ini telah diterjemahkan menggunakan perkhidmatan terjemahan AI Co-op Translator (https://github.com/Azure/co-op-translator). Walaupun kami berusaha untuk ketepatan, sila ambil maklum bahawa terjemahan automatik mungkin mengandungi ralat atau ketidaktepatan. Dokumen asal dalam bahasa asalnya hendaklah dianggap sebagai sumber rujukan yang muktamad. Untuk maklumat yang kritikal, terjemahan profesional oleh penterjemah manusia disyorkan. Kami tidak bertanggungjawab terhadap sebarang salah faham atau salah tafsiran yang timbul daripada penggunaan terjemahan ini.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
